In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import train_test_split

In [2]:
metadata = pd.read_csv('meta_orbitofrontal_cortex.csv', sep=';', index_col=0)

In [3]:
metadata

,sample_id,age,disease,sex,smoking
GSM,,,,,
GSM8037248,s72,64,1,Male,Never
GSM8037249,s33,51,0,Female,Never
GSM8037250,s32,59,1,Male,Current
GSM8037251,s80,45,0,Female,Never
GSM8037253,s44,41,1,Male,Current
...,...,...,...,...,...
GSM8037339,s40,62,1,Male,Current
GSM8037340,s79,42,1,Female,Current
GSM8037341,s31,60,1,Female,Never


In [ ]:
df = pd.read_csv('train_methylation_orbitofrontal_cortex.csv', sep=';', index_col=0)

In [ ]:
df.head(5)

In [ ]:
samples = X.index.intersection(metadata.index)

In [ ]:
X = X.loc[samples]
y = metadata.loc[samples, 'age']

In [ ]:
X = X.astype('float32')

In [ ]:
y

In [ ]:
X.isnull().sum()

In [ ]:
y.isnull().sum()

In [ ]:
def horvath_transform(age, adult_age=20):
    age = np.array(age)
    return np.where(age <= adult_age, np.log(age+1)-np.log(adult_age + 1), (age - adult_age)/(adult_age+1))

In [ ]:
def horvath_inverse(transformed_age, adult_age=20):
    transformed_age = np.array(transformed_age)
    return np.where(transformed_age < 0, np.exp(transformed_age + np.log(adult_age+1))-1, transformed_age * (adult_age+1) + adult_age)

In [ ]:
y_transformed = horvath_transform(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression

def perform_feature_selection(X, y, k=50000):
    selector = SelectKBest(score_func=f_regression, k=k)
    X_selected = selector.fit_transform(X, y)
    
    selected_features = X.columns[selector.get_support()]
    
    print(f"Final feature count: {X_selected.shape[1]}")
    return X_selected, selected_features

X_reduced, clock_sites = perform_feature_selection(X_train, y_train, k=150000)

In [ ]:
X_train_reduced = X_reduced
X_test_reduced = X_test[clock_sites]

In [ ]:
model = ElasticNetCV(
    l1_ratio=[0.1, 0.5, 0.7, 0.9],
    cv=5,
    alphas=50,
    n_jobs=8,
    max_iter=10000,
    random_state=42
)
model.fit(X_train_reduced, y_train)

In [ ]:
preds_transformed = model.predict(X_test_reduced)
preds_years = horvath_inverse(preds_transformed)
actual_years = horvath_inverse(y_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(actual_years, preds_years)
r2 = r2_score(actual_years, preds_years)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.scatter(actual_years, preds_years, alpha=0.6, color='teal')
plt.plot([0, 100], [0, 100], 'r--')
plt.title(f"Orb. Cortex Clock: MAE = {mae:.2f} years (R² = {r2:.2f})")
plt.xlabel("Chronological Age")
plt.ylabel("Predicted DNAm Age")
plt.show()